# 🚀 Semiconductor Image Restoration (NAFNet-SR) - Maximum Performance Colab Notebook

This notebook trains **NAFNet-SR** fully harnessing **Tesla T4 Tensor Cores** using Automatic Mixed Precision (AMP FP16), cuDNN benchmarking, and multi-threaded data loading.

### **GPU Optimizations Enabled**
- **Tensor Core Acceleration**: PyTorch Automatic Mixed Precision (`AMP FP16` with `GradScaler`)
- **cuDNN Autotuner**: `torch.backends.cudnn.benchmark = True`
- **Asynchronous Memory Transfers**: `pin_memory=True`, `non_blocking=True`
- **Multi-Worker Data Pipeline**: `num_workers=4`, `persistent_workers=True`
- **High Throughput Batching**: `batch_size=32`


## Step 1: Verify Tesla T4 GPU & Tensor Cores


In [ ]:
!nvidia-smi


## Step 2: Clone Repository (Branch `Kunal`)


In [ ]:
!git clone -b Kunal https://github.com/KJ-CORE/semicon_2026.git
%cd semicon_2026


## Step 3: Install Dependencies


In [ ]:
!pip install -r requirements.txt


## Step 4: Extract Dataset & Create Validation Split
*Mounts Google Drive to extract `train.zip` & `Test_NoisyLR.zip` automatically without re-uploading every session.*

In [ ]:
import os, zipfile, glob, shutil, random
from google.colab import drive

# Mount Google Drive for permanent dataset storage
drive.mount('/content/drive')

# Check for dataset in Drive or local directory
drive_train_zip = '/content/drive/MyDrive/train.zip'
local_train_zip = 'train.zip'
zip_path = drive_train_zip if os.path.exists(drive_train_zip) else (local_train_zip if os.path.exists(local_train_zip) else None)

if zip_path:
    print(f'Extracting dataset from: {zip_path}...')
    os.makedirs('data', exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall('data')
    print('Training dataset extraction complete!')
else:
    print('⚠️ train.zip not found! Please upload train.zip to your Google Drive MyDrive root.')

# Check for test set
drive_test_zip = '/content/drive/MyDrive/Test_NoisyLR.zip'
local_test_zip = 'Test_NoisyLR.zip'
test_zip_path = drive_test_zip if os.path.exists(drive_test_zip) else (local_test_zip if os.path.exists(local_test_zip) else None)

if test_zip_path:
    print(f'Extracting test dataset from: {test_zip_path}...')
    os.makedirs('data/test', exist_ok=True)
    with zipfile.ZipFile(test_zip_path, 'r') as z:
        z.extractall('data/test')
    print('Test dataset extraction complete!')

# Create Train / Val split (10% validation)
if os.path.exists('data/train/NoisyLR') and not os.path.exists('data/val'):
    random.seed(42)
    os.makedirs('data/val/NoisyLR', exist_ok=True)
    os.makedirs('data/val/GT', exist_ok=True)
    files = sorted(glob.glob('data/train/NoisyLR/*.npy'))
    val_files = random.sample(files, k=int(len(files) * 0.1))
    for f in val_files:
        fname = os.path.basename(f)
        shutil.move(f, os.path.join('data/val/NoisyLR', fname))
        shutil.move(os.path.join('data/train/GT', fname), os.path.join('data/val/GT', fname))
    print(f'Created Val Split: {len(val_files)} samples moved to data/val/')


## Step 5: Start Maximum Compute Model Training (Tensor Cores + AMP FP16)
Harnesses Tesla T4 320 Tensor Cores with AMP FP16, `batch_size=32`, `num_workers=4`, and `patch_size=64`.

In [ ]:
!python train.py --epochs 200 --batch_size 32 --lr 3e-4 --scale 2 --patch_size 64 --num_workers 4


## Step 6: Backup Checkpoint to Google Drive


In [ ]:
!cp weights/best_model.pt /content/drive/MyDrive/best_model.pt
print('Best model checkpoint saved permanently to Google Drive!')


## Step 7: Run High-Speed Evaluation / Inference


In [ ]:
!python eval.py --input_dir data/test/NoisyLR --output_dir data/output_restored --weights weights/best_model.pt --scale 2
